## 4.2 NLP(ish)

https://maartengr.github.io/BERTopic/getting_started/quickstart/quickstart.html

## décider de comment on découpe
- sentence ?
- paragraphe (existe pas ici) -> mais possible avec wtpsplit : https://github.com/segment-any-text/wtpsplit
- dire que tant pis ? Ok si utilise modèle avec fenetre suffisante (qwen ? https://huggingface.co/Qwen/Qwen3-Embedding-0.6B)
- faire un check global du nb token par intervention et cut que ce qui dépasse ?
- Passer à l'échelle de la phrase pour tout ce qui va être pour sentiment, réseau de mots, proba des termes, etc.
- etc.

In [ ]:
# TODO: remplacer texte_clean par texte (corrigé dans nb3)
# TODO: aviser si vire id_orateur et utiliser id_acteur partout

# TODO: remove unused imports when finalized
import pandas as pd
import spacy
# import nltk
# from nltk.corpus import stopwords

# import bertopic
from bertopic import BERTopic

# from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

# %pip install einops
# !python -m spacy download fr_core_news_sm

In [2]:
df = pd.read_csv(
    "../data/interim/df_repu.csv", low_memory=False, dtype={"ID_orateur": str}
)

In [3]:
df["DateSeance_ts"] = pd.to_datetime(df["DateSeance"], format="%Y%m%d%H%M%S%f")
df["DateSeance_day"] = df["DateSeance_ts"].dt.normalize()  # guess it works


## Bert things

In [4]:
#########
# Si besoin, revenir au plus simple :
#########

# nltk.download("stopwords")
# french_stopwords = list(set(stopwords.words("french")))
# vectorizer_model = CountVectorizer(stop_words=french_stopwords)
# topic_model = bertopic.BERTopic(language="french", vectorizer_model=vectorizer_model)
# # topics, probs = topic_model.fit_transform(df["Texte_clean"])

In [5]:
# TODO: redo with the good sentencetransformer

Stopwords :
- https://maartengr.github.io/BERTopic/getting_started/tips_and_tricks/tips_and_tricks.html#document-length
- Instead, we can use the CountVectorizer to preprocess our documents after having generated embeddings and clustered our documents. Personally, I have found almost no disadvantages to using the CountVectorizer to remove stopwords and it is something I would strongly advise to try out:
- We can also use the ClassTfidfTransformer to reduce the impact of frequent words. The end result is very similar to explicitly removing stopwords but this process does this automatically:

In [ ]:
# TODO: Tester Flaubert et autres, qwen, etc. ALibaba = cry in GPU, , etc. Aviser dans colab ou humanum ?
# Qwen pour sa Context Length ? + est multilingue ?
# pousser vers leur 4B ou 8B si ressources suffisantes ?
# ou https://huggingface.co/jinaai/jina-embeddings-v3

# "dangvantuan/sentence-camembert-large"
# "all-MiniLM-L6-v2"
# Maybe https://huggingface.co/jinaai/jina-embeddings-v3

# embedding_model = SentenceTransformer("jinaai/jina-embeddings-v3", trust_remote_code=True)

#######
# ICI
#######
# Les tests à l'arrache active tigger donnent des trucs pas mal avec
# "Alibaba-NLP/gte-multilingual-base"
# https://huggingface.co/Alibaba-NLP/gte-multilingual-base


In [ ]:
# vectorizer_model et french stopwords
# Avec spacy
nlp = spacy.load("fr_core_news_sm")  # !python -m spacy download fr_core_news_sm
french_stopwords = list(nlp.Defaults.stop_words)
vectorizer_model = CountVectorizer(stop_words=french_stopwords)

# # Ou avec nltk
# import nltk
# from nltk.corpus import stopwords
# nltk.download("stopwords")
# french_stopwords = list(set(stopwords.words("french")))
# vectorizer_model = CountVectorizer(stop_words=french_stopwords)


#####################################
# TODO: Choisir le modèle d'embedding final
#####################################


embedding_model = SentenceTransformer(
    # "Alibaba-NLP/gte-multilingual-base", # cry in gpu, something wrong ?
    # "dangvantuan/sentence-camembert-large",
    "all-MiniLM-L6-v2",  # celui par défaut ?
    # "jinaai/jina-embeddings-v3",
    trust_remote_code=True,
)

print(
    "device used :", embedding_model.device
)  # Vérifie si le modèle est sur GPU ou CPU

# ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

# créer le modèle
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,  # remove stopwords after embbedings
    # ctfidf_model=ctfidf_model, # (or) reduce the impact of frequent word
)

device used : mps:0


In [9]:
# Fiter le modèle
topics, probs = topic_model.fit_transform(
    df["Texte_clean"]
)  # .tolist() ? pas obligatoire ?

# Test taille batch

In [10]:
# Reste quand même trop gros, faudrait couper les interventions sans doute.

# # from sentence_transformers import SentenceTransformer
# import numpy as np

# # embedding_model = SentenceTransformer("jinaai/jina-embeddings-v3", trust_remote_code=True)
# texts = df["Texte_clean"].tolist()

# batch_size = 8
# embeddings = []

# for i in range(0, len(texts), batch_size):
#     batch_texts = texts[i:i+batch_size]
#     batch_embeds = embedding_model.encode(batch_texts, show_progress_bar=False)
#     embeddings.append(batch_embeds)

# embeddings = np.vstack(embeddings)

# topics, probs = topic_model.fit_transform(df["Texte_clean"], embeddings=embeddings)


In [11]:
# # V1 safetensors
# topic_model.save(
#     "../models/bert/camembert_safetensors",
#     serialization="safetensors",
#     save_ctfidf=True,
#     save_embedding_model=embedding_model,
# )

# # V2 pytorch
# topic_model.save(
#     "../models/bert/camembert_pytorch",
#     serialization="pytorch",
#     save_ctfidf=True,
#     save_embedding_model=embedding_model,
# )

# # V3 pickle (pas recommandé ?)
# topic_model.save("../models/bert/############", save_embedding_model=True)


In [12]:
# Load from directory
# loaded_model = BERTopic.load("../models/bert/############")


In [13]:
# TODO: does mean pooling thing really works ?

Pour "dangvantuan/sentence-camembert-large"
- No sentence-transformers model found with name dangvantuan/sentence-camembert-large. Creating a new one with MEAN pooling.
- https://github.com/UKPLab/sentence-transformers/issues/2779 In short: even with a will create a new model with mean pooling warning, your model uses the tokenizer from https://huggingface.co/dangvantuan/sentence-camembert-large.


DOES IT REALLY WORKS ?

In [14]:
# help(topic_model)

In [15]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,7026,-1_qu_république_loi_été,"[qu, république, loi, été, bien, faire, état, ...","[Monsieur le président, monsieur le président ..."
1,0,605,0_euros_milliards_budget_finances,"[euros, milliards, budget, finances, millions,...","[Monsieur le président, monsieur le ministre d..."
2,1,412,1_territoires_communes_territoriale_collectivités,"[territoires, communes, territoriale, collecti...","[Monsieur le président, monsieur le secrétaire..."
3,2,370,2_république_ve_vive_exemplaire,"[république, ve, vive, exemplaire, êtes, vie, ...","[La République, c'est lui !, « La République, ..."
4,3,317,3_associations_engagement_contrat_association,"[associations, engagement, contrat, associatio...",[On aurait pu penser que les arguments allaien...
...,...,...,...,...,...
85,84,12,84_caméras_images_piétons_police,"[caméras, images, piétons, police, policiers, ...",[Avant de souligner les interrogations que sus...
86,85,12,85_laïcité_laïque_spirituelles_liberté,"[laïcité, laïque, spirituelles, liberté, croir...","[Tout d'abord, je tiens à rappeler que nous pa..."
87,86,11,86_islamiste_paty_samuel_islamisme,"[islamiste, paty, samuel, islamisme, islamiste...",[Après l'horreur de la décapitation du profess...
88,87,10,87_réservistes_salaire_enseignants_logements,"[réservistes, salaire, enseignants, logements,...","[Non.Pour réunir les fonds, ils doivent cherch..."


### Tester des trucs rapides

In [16]:
# # Les principales fonctions à tester pour avoir un aperçu simple :

# topic_model.get_topic_info()
# topic_model.visualize_barchart()
# topic_model.visualize_topics()
# topic_model.visualize_hierarchy()
# topic_model.visualize_documents(df["Texte_clean"].to_list())


### Tester plus

In [17]:
# Optional: visualize
fig_topic_distance_map = topic_model.visualize_topics()
fig_topic_distance_map

In [ ]:
fig_topic_distance_map.write_html("../reports/figures/topic_distance_map.html")

In [18]:
table_topic = topic_model.get_topic_info()
table_topic[:20]

,Topic,Count,Name,Representation,Representative_Docs
0,-1,7026,-1_qu_république_loi_été,"[qu, république, loi, été, bien, faire, état, ...","[Monsieur le président, monsieur le président ..."
1,0,605,0_euros_milliards_budget_finances,"[euros, milliards, budget, finances, millions,...","[Monsieur le président, monsieur le ministre d..."
2,1,412,1_territoires_communes_territoriale_collectivités,"[territoires, communes, territoriale, collecti...","[Monsieur le président, monsieur le secrétaire..."
3,2,370,2_république_ve_vive_exemplaire,"[république, ve, vive, exemplaire, êtes, vie, ...","[La République, c'est lui !, « La République, ..."
4,3,317,3_associations_engagement_contrat_association,"[associations, engagement, contrat, associatio...",[On aurait pu penser que les arguments allaien...
5,4,288,4_violences_violence_ordre_forces,"[violences, violence, ordre, forces, policiers...","[Monsieur le président, monsieur le vice-prési..."
6,5,241,5_constitution_amendement_article_constitutionnel,"[constitution, amendement, article, constituti...",[Nos collègues du groupe Les Républicains prop...
7,6,231,6_républicain_républicains_républicaine_arc,"[républicain, républicains, républicaine, arc,...","[Ce n'est pas républicain !, Voilà un républic..."
8,7,222,7_police_sécurité_gendarmerie_policiers,"[police, sécurité, gendarmerie, policiers, for...","[Monsieur le président, monsieur le ministre, ..."
9,8,216,8_enfants_enfant_parents_instruction,"[enfants, enfant, parents, instruction, famill...","[Madame la présidente, monsieur le ministre, m..."


In [103]:
table_topic.to_csv("../data/interim/table_topics.csv", index=False)

In [97]:
df["Topic"] = topics

In [ ]:
import csv

df.to_csv("../data/interim/df_repu_with_topics.csv", index=False, quoting=csv.QUOTE_ALL)

## Dynamic topic model

* `global_tuning`
  * Whether to average the topic representation of a topic at time *t* with its global topic representation
* `evolution_tuning`
  * Whether to average the topic representation of a topic at time *t* with the topic representation of that topic at time *t-1*
* `nr_bins`
  * The number of bins to put our timestamps into. It is computationally inefficient to extract the topics at thousands of different timestamps. Therefore, it is advised to keep this value below 20.


In [22]:
topics_over_time = topic_model.topics_over_time(
    docs=df["Texte_clean"],
    timestamps=df["DateSeance_ts"],
    global_tuning=True,
    evolution_tuning=True,
    nr_bins=20,
)

In [23]:
fig_dynamic_topic = topic_model.visualize_topics_over_time(
    topics_over_time, top_n_topics=10
)
fig_dynamic_topic

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hoverinfo': 'text',
              'hovertext': [<b>Topic 0</b><br>Words: école, enseignement,
                            éducation, enfants, élèves, <b>Topic 0</b><br>Words:
                            école, élèves, éducation, enseignement, enfants,
                            <b>Topic 0</b><br>Words: école, éducation, élèves,
                            enfants, écoles, <b>Topic 0</b><br>Words: école,
                            élèves, enfants, éducation, handicap, <b>Topic
                            0</b><br>Words: école, enfants, élèves, éducation,
                            enseignement, <b>Topic 0</b><br>Words: école, enfants,
                            éducation, enfance, jeunes, <b>Topic 0</b><br>Words:
                            école, élèves, enfants, éducation, enseignement,
                            <b>Topic 0</b><br>Words: handicap, école, élèves,
                            enfants, éducation, <b>Topic 0</b><br>Words: école,
                            enfants, directeurs, élèves, parents, <b>Topic
                            0</b><br>Words: école, éducation, élèves, enfants,
                            enseignants, <b>Topic 0</b><br>Words: école, enfants,
                            instruction, éducation, famille, <b>Topic
                            0</b><br>Words: enfants, école, enfant, instruction,
                            famille, <b>Topic 0</b><br>Words: école, élèves,
                            directeurs, éducation, enseignement, <b>Topic
                            0</b><br>Words: école, enfants, élèves, éducation,
                            scolaire, <b>Topic 0</b><br>Words: école, handicap,
                            élèves, enfants, rentrée, <b>Topic 0</b><br>Words:
                            école, élèves, éducation, scolaire, uniforme, <b>Topic
                            0</b><br>Words: élèves, école, éducation, enfants,
                            enseignants, <b>Topic 0</b><br>Words: école, ensa, opp,
                            handicap, élèves, <b>Topic 0</b><br>Words: école,
                            élèves, parents, éducation, enfants, <b>Topic
                            0</b><br>Words: école, élèves, enfants, éducation,
                            enseignement],
              'marker': {'color': '#E69F00'},
              'mode': 'lines',
              'name': '0_école_élèves_enfants_éducation',
              'type': 'scatter',
              'x': array(['2017-06-26T02:08:09.600000000', '2017-11-02T10:12:00.000000000',
                          '2018-03-09T05:24:00.000000000', '2018-07-14T00:36:00.000000000',
                          '2018-11-17T19:48:00.000000000', '2019-03-24T15:00:00.000000000',
                          '2019-07-29T10:12:00.000000000', '2019-12-03T05:24:00.000000000',
                          '2020-04-08T00:36:00.000000000', '2020-08-12T19:48:00.000000000',
                          '2020-12-17T15:00:00.000000000', '2021-04-23T10:12:00.000000000',
                          '2021-08-28T05:24:00.000000000', '2022-01-02T00:36:00.000000000',
                          '2022-05-08T19:48:00.000000000', '2022-09-12T15:00:00.000000000',
                          '2023-01-17T10:12:00.000000000', '2023-05-24T05:24:00.000000000',
                          '2023-09-28T00:36:00.000000000', '2024-02-01T19:48:00.000000000'],
                         dtype='datetime64[ns]'),
              'y': {'bdata': 'CAA6AEUAHQCkACMAIgAUACAAJgC8ADkAKgATAAsAVAAyABQAaABGAA==', 'dtype': 'i2'}},
             {'hoverinfo': 'text',
              'hovertext': [<b>Topic 1</b><br>Words: euros, impôt, budget,
                            milliards, millions, <b>Topic 1</b><br>Words: euros,
                            milliards, finances, impôt, budget, <b>Topic
                            1</b><br>Words: euros, finances, budgétaire, milliards,
                            budget, <b>Topic 1</b><br>Words: euros, budget,
                            finances, milliards, m

In [ ]:
fig_dynamic_topic.write_html("../reports/figures/dynamic_topics.html")

## Hierarchical topics

In [32]:
hierarchical_topics = topic_model.hierarchical_topics(df["Texte_clean"])

100%|██████████| 71/71 [00:00<00:00, 692.89it/s]


In [ ]:
fig_hierarchical = topic_model.visualize_hierarchy(
    hierarchical_topics=hierarchical_topics
)
fig_hierarchical

In [ ]:
fig_hierarchical.write_html("../reports/figures/hierarchical_topics.html")

In [40]:
tree = topic_model.get_topic_tree(hierarchical_topics)
print(tree)

.
├─république_républicain_arc_républicains_honte
│    ├─républicain_arc_républicains_république_grâce
│    │    ├─■──arc_républicain_sortir_étonnement_situerait ── Topic: 58
│    │    └─républicain_républicains_république_grâce_iiie
│    │         ├─■──républicain_république_iiie_ordre_barrage ── Topic: 16
│    │         └─■──républicains_grâce_républicain_attachées_espèrent ── Topic: 40
│    └─république_honte_ve_aimez_vie
│         ├─république_honte_ve_vie_êtes
│         │    ├─■──république_honte_êtes_détruisant_détruisez ── Topic: 20
│         │    └─■──ve_république_vie_présidente_pen ── Topic: 30
│         └─■──aimez_aiment_république_rance_aimait ── Topic: 67
└─qu_république_loi_été_france
     ├─qu_loi_république_france_bien
     │    ├─calédonie_nouvelle_49_ve_république
     │    │    ├─droite_extrême_gauche_peur_histoire
     │    │    │    ├─■──extrême_droite_immigration_peur_ennemi ── Topic: 71
     │    │    │    └─■──droite_extrême_gauche_histoire_dos ── Topic: 39
    

## Topic reduction

In [ ]:
# topics_to_merge = [[X, Y],
#                    [Z, W]]
# topic_model.merge_topics(df["Texte_clean"], topics_to_merge)

In [ ]:
# DONT : # topic_model.reduce_topics(df["Texte_clean"], nr_topics=40) # DONT, IT NUKES THE TOPICS IN MODEL
# # Access updated topics
# topics = topic_model.topics_

2025-07-02 18:46:30,812 - BERTopic - Topic reduction - Reducing number of topics
2025-07-02 18:46:30,813 - BERTopic - Topic reduction - Reduced number of topics from 20 to 20


### Topics per class

In [52]:
df.columns

Index(['UID', 'SeanceRef', 'SessionRef', 'DateSeance', 'DateSeanceJour',
       'NumSeanceJour', 'NumSeance', 'TypeAssemblee', 'Legislature', 'Session',
       'NomFichierJO', 'President', 'Titre_general', 'Sous_titre',
       'Contexte_hierarchique', 'Section_courante', 'Sujet_point',
       'Valeur_ODJ', 'Point_ID', 'Point_type', 'ID_paragraphe', 'Ordre_seance',
       'Code_grammaire', 'Code_style', 'Code_parole', 'Role_debat',
       'Nom_orateur', 'Qualite_orateur', 'ID_orateur', 'stime', 'Texte',
       'Source_extraction', 'len_dirtytext', 'legislatureLast', 'civ', 'nom',
       'prenom', 'villeNaissance', 'naissance', 'age', 'groupe', 'groupeAbrev',
       'departementNom', 'departementCode', 'circo', 'datePriseFonction',
       'job', 'mail', 'twitter', 'facebook', 'website', 'nombreMandats',
       'experienceDepute', 'scoreParticipation',
       'scoreParticipationSpecialite', 'scoreLoyaute', 'scoreMajorite',
       'active', 'dateMaj', 'Texte_clean', 'repu_match_valide',
  

In [74]:
df["groupeAbrev"] = df["groupeAbrev"].fillna("gouv_TEMP")

In [ ]:
# ATTENTION, PLANTAIT À CAUSE DES NA
topics_per_class = topic_model.topics_per_class(
    df["Texte_clean"], classes=df["groupeAbrev"]
)

In [ ]:
fig_topics_per_class = topic_model.visualize_topics_per_class(
    topics_per_class, top_n_topics=10
)
fig_topics_per_class

In [ ]:
fig_topics_per_class.write_html("../reports/figures/topics_per_class.html")

In [ ]:
# TODO: revoir regroupement des topics
# TODO: revoir Topic distribution
# TODO: aviser genAI sur le nom des topics ?

In [105]:
df.shape

(3668, 64)

In [106]:
df["Texte_clean"].str.len().describe()

count     3668.000000
mean      1455.530807
std       1701.023067
min         18.000000
25%        409.750000
50%        926.000000
75%       1856.000000
max      20908.000000
Name: Texte_clean, dtype: float64